# 04 · REVEL & PrimateAI — supervised meta-predictors and the *circularity* problem

**Toolkit notebook 4 of the CFTR variant-effect series.** Audience: bench scientists new to
variant-effect prediction.

In notebook 03 we met the **unsupervised** tools (AlphaMissense, EVE, ESM1b) — models that never
saw a single clinical label. This notebook covers two tools built the *other* way:

- **REVEL** — a **supervised** machine-learning *ensemble* that was **trained on known
  pathogenic/benign variants**.
- **PrimateAI** — a **semi-supervised** deep network trained on a clever *proxy* for benignity.

Both are powerful. But because REVEL learned from clinical labels, comparing it back to those same
labels (ClinVar) is **partly circular** — and learning to spot that trap is the whole point of this
notebook.

---

> ## ⚠️ DEMO DATA — READ THIS FIRST
>
> The REVEL and PrimateAI scores in this notebook are **DEMO values**: a tiny, **hand-authored**
> table of ~13 curated CFTR variants baked into `toolkit.py`. **They are NOT the real output of
> REVEL or PrimateAI.** They exist only so the code runs end-to-end for teaching.
>
> Every table below keeps a **`source`** column so you can never mistake DEMO for REAL. Section 6
> ("How to get REAL data") tells you exactly where to download the genuine genome-wide scores.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 2 · REVEL — what is it?

**REVEL** = *Rare Exome Variant Ensemble Learner* (Ioannidis et al. 2016, *American Journal of
Human Genetics*, **PMID 27666373**).

Think of REVEL as a **committee vote of 13 other predictors**. Instead of inventing a brand-new way
to judge a missense variant, its authors took the scores of **13 individual tools** (e.g. SIFT,
PolyPhen-2, MutationTaster, and several conservation scores) and trained a **random forest** — a
supervised machine-learning model — to combine them into one number.

Key facts to remember:

| Property | REVEL |
|---|---|
| Learning type | **Supervised** (learned from labelled examples) |
| Model | Random-forest **ensemble** of 13 component predictors |
| Trained on | Curated **pathogenic** and **benign** missense variants |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Covers | Missense variants only |

The "supervised" part is crucial: someone had to **hand REVEL a training set of variants already
labelled pathogenic or benign**. Where did those labels come from? Databases like **ClinVar** and
**HGMD**. Hold that thought — it is the seed of the circularity problem in section 3.


In [2]:
# Load the DEMO REVEL table. Note the `source` column says DEMO on every row.
revel = tk.load_revel()
print("REVEL score range in this DEMO table:", revel.revel_score.min(), "→", revel.revel_score.max())
revel

REVEL score range in this DEMO table: 0.1 → 0.94


,protein_variant,revel_score,source
0,G551D,0.940,DEMO
2,R117H,0.520,DEMO
3,R334W,0.880,DEMO
4,G85E,0.900,DEMO
5,D1152H,0.672,DEMO
6,R668C,0.220,DEMO
7,Tyr161Cys,0.872,DEMO
8,Gly970Asp,0.812,DEMO
9,Ser912Leu,0.782,DEMO
10,Val520Phe,0.755,DEMO


## 3 · ⚠️ THE CIRCULARITY WARNING — the key idea in this notebook

> ### 🔁 Why "REVEL disagrees with ClinVar" can be *misleading*
>
> REVEL was **trained on labels that share lineage with ClinVar and HGMD.**
>
> So when you benchmark REVEL against ClinVar and find they **agree**, that agreement may be
> **partly baked in** — REVEL may simply be *remembering* variants it was trained on. This is
> **label leakage**: information from the "answer key" leaked into the model.
>
> And when REVEL **disagrees** with ClinVar, that disagreement is **not guaranteed to be
> independent evidence** either — it can reflect quirks of the training labels rather than a fresh,
> orthogonal opinion.
>
> **Bottom line:** a supervised predictor benchmarked against its own training-label source is
> **partly circular**. It looks more accurate than it truly is on *new* variants.

**Contrast with the unsupervised tools from notebook 03:**

| Tool | Learned from clinical labels? | Benchmarking vs ClinVar is… |
|---|---|---|
| AlphaMissense, EVE, ESM1b | **No** (sequence / structure / evolution only) | **Fair-ish** — an independent opinion |
| **REVEL** | **Yes** (ClinVar/HGMD lineage) | **Partly circular** — beware label leakage |
| PrimateAI | Partly (a benign *proxy*, not clinical labels) | **Medium** circularity |

➡️ **Forward reference:** Notebook **08** builds the de-circularization / benchmarking analysis
properly — using **orthogonal** truth sets like CFTR2 functional data and population frequency, so
supervised tools are not simply graded against their own homework.


## 4 · REVEL isn't one cut-point — it's a *graded* scale

A very common shortcut is: *"REVEL ≥ 0.75 → likely pathogenic, otherwise not."* That single cut
throws away information.

The ClinGen **Sequence Variant Interpretation** working group (**Pejaver et al. 2022**,
*AJHG*, **PMID 36413997**) *calibrated* REVEL against the **ACMG/AMP** evidence framework. They
showed a REVEL score maps to **graded tiers of pathogenic evidence strength** (Supporting →
Moderate → Strong), not a single yes/no line:

| REVEL score ≥ | ACMG pathogenic evidence strength (approx.) |
|---|---|
| **0.932** | **Strong** (PP3_Strong) |
| **0.773** | **Moderate** (PP3_Moderate) |
| **0.644** | **Supporting** (PP3_Supporting) |
| **0.290** | *below this* → **Benign** supporting evidence (BP4) |

(These break-points are approximate and are the ones used in the toolkit's teaching table.)

**Why it matters:** two variants at REVEL 0.65 and 0.95 are *both* "≥ 0.75-ish territory" under a
single cut, but the calibration says one is only **Supporting** evidence and the other is
**Strong**. Collapsing them to one label loses exactly the nuance a curator needs.

The toolkit deliberately ships a **single** binary cut (`tk.THRESHOLDS['revel']`) to keep the
`call_from_score` helper simple — but you should know the richer, graded reality exists.


In [3]:
# The toolkit's SIMPLE binary cut-points (one 'pathogenic' line, one 'benign' line):
tk.THRESHOLDS['revel']

{'path': 0.75, 'benign': 0.29}

## 5 · PrimateAI — the *semi-supervised* alternative

**PrimateAI** (Sundaram et al. 2018, *Nature Genetics*, **PMID 30038395**) is a **deep neural
network**, but it dodges the "we need labelled pathogenic variants" problem with a neat trick.

Instead of training on curated *pathogenic* labels, it trains on a **proxy for benignity**:

> A missense variant that is **common** in healthy humans — or is seen as the normal amino acid in
> **other primates** (chimp, gorilla, …) — is very likely **tolerated / benign**. Evolution has
> already "tested" it.

So PrimateAI learns from **hundreds of thousands of common human & non-human-primate missense
variants** as its benign class. This is why it's called **semi-supervised**: it uses labels, but
those labels are a *population-frequency proxy*, **not** clinical pathogenic/benign assertions.

| Property | PrimateAI |
|---|---|
| Learning type | **Semi-supervised** (benign proxy from primate/common variants) |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Pathogenic cut | **≥ 0.803** |
| Circularity vs ClinVar | **Medium** — it never trained on ClinVar *pathogenic* labels, but common-variant proxies can still overlap benign ClinVar entries |


In [4]:
tk.THRESHOLDS['primate_ai']

{'path': 0.803, 'benign': 0.483}

## 6 · How to get the REAL scores (not this DEMO table)

The values above are illustrative. To do genuine CFTR analysis, download the real genome-wide
predictions and **join them onto your variants by genomic coordinate**:

**REVEL**
- Download the **genome-wide REVEL table** from
  `https://sites.google.com/site/revelgenomics/`.
- It is keyed by **genomic coordinate**: `chromosome, position, ref, alt` (GRCh37/38).
- Join it to your gnomAD/ClinVar variants on `(chr, pos, ref, alt)`, **not** on the protein change,
  because one protein change can arise from several codon changes.

**PrimateAI**
- Scores are distributed by **Illumina** (PrimateAI / PrimateAI-3D). You accept a data-use
  agreement, then download the precomputed score files.
- Also keyed by genomic coordinate — join the same way.

> ⚠️ **Coordinate gotcha for CFTR:** CFTR sits on the **minus strand** of chromosome 7. When you
> join on `(chr, pos, ref, alt)`, make sure your ref/alt alleles are on the **same genomic strand**
> as the downloaded table, or you will silently match nothing. (The toolkit's CADD helper shows the
> strand-flip trick.)


## 7 · Load both DEMO tables, merge, and make calls

Now the hands-on part. We:
1. load the DEMO REVEL and PrimateAI tables,
2. **merge** them on `protein_variant`,
3. use `tk.call_from_score(...)` to turn each raw score into a pathogenic / uncertain / benign
   **call** using the simple binary thresholds.


In [5]:
revel     = tk.load_revel()        # protein_variant, revel_score, source
primateai = tk.load_primateai()    # protein_variant, primate_ai_score, source

# Merge on the shared protein_variant key. Keep ONE source column (both say DEMO).
merged = revel.merge(
    primateai.drop(columns='source'),
    on='protein_variant', how='inner'
)

# Turn each raw score into a 3-class call via the toolkit's single scoring helper.
merged['revel_call'] = merged['revel_score'].apply(lambda s: tk.call_from_score(s, 'revel'))
merged['pai_call']   = merged['primate_ai_score'].apply(lambda s: tk.call_from_score(s, 'primate_ai'))

# Present the columns the task asks for, in a readable order.
result = merged[['protein_variant', 'revel_score', 'revel_call',
                 'primate_ai_score', 'pai_call', 'source']]
result

,protein_variant,revel_score,revel_call,primate_ai_score,pai_call,source
0,G551D,0.940,pathogenic,0.930,pathogenic,DEMO
1,R117H,0.520,uncertain,0.440,benign,DEMO
2,R334W,0.880,pathogenic,0.850,pathogenic,DEMO
3,G85E,0.900,pathogenic,0.880,pathogenic,DEMO
4,D1152H,0.672,uncertain,0.600,uncertain,DEMO
5,R668C,0.220,benign,0.280,benign,DEMO
6,Tyr161Cys,0.872,pathogenic,0.841,pathogenic,DEMO
7,Gly970Asp,0.812,pathogenic,0.782,uncertain,DEMO
8,Ser912Leu,0.782,pathogenic,0.751,uncertain,DEMO
9,Val520Phe,0.755,pathogenic,0.731,uncertain,DEMO


Notice how the two tools **mostly agree** but not always — a variant can sit above REVEL's 0.75
line yet below PrimateAI's stricter 0.803 line, landing in *pathogenic* for one and *uncertain* for
the other. Disagreements like these are exactly what a careful curator investigates rather than
averaging away.


## 8 · A taste of the graded idea — binning REVEL into the 4 Pejaver tiers

Instead of one line at 0.75, let's sort the DEMO variants into the **four calibrated tiers** from
section 4 using `pandas.cut`. This is a miniature version of what ACMG-style graded evidence looks
like in practice.


In [6]:
# Edges from the Pejaver 2022 calibration (approx.). -inf/​+inf capture the ends.
tier_edges  = [-np.inf, 0.290, 0.644, 0.773, 0.932, np.inf]
tier_labels = [
    'Benign-supporting (<0.290)',
    'Indeterminate (0.290–0.644)',
    'Path Supporting (0.644–0.773)',
    'Path Moderate (0.773–0.932)',
    'Path Strong (≥0.932)',
]

merged['revel_tier'] = pd.cut(merged['revel_score'], bins=tier_edges,
                              labels=tier_labels, right=False)

# How many DEMO variants land in each graded tier?
tier_counts = merged['revel_tier'].value_counts().reindex(tier_labels).fillna(0).astype(int)
print("DEMO REVEL variants per Pejaver 2022 evidence tier:\n")
print(tier_counts.to_string())

DEMO REVEL variants per Pejaver 2022 evidence tier:

revel_tier
Benign-supporting (<0.290)       2
Indeterminate (0.290–0.644)      1
Path Supporting (0.644–0.773)    4
Path Moderate (0.773–0.932)      5
Path Strong (≥0.932)             1


Even in this tiny DEMO set you can see variants spread across **several** evidence strengths — the
detail a single 0.75 cut-point would flatten into just "pathogenic vs not".


## 9 · What the toolkit *records* about each tool

`toolkit.py` keeps a **registry** of metadata for every predictor — its learning type, its signal,
and (critically) its **circularity** rating. Printing it for REVEL and PrimateAI reinforces
everything above and is what notebook 08 uses to decide which tools can be fairly benchmarked
against ClinVar.


In [7]:
for name in ['REVEL', 'PrimateAI']:
    info = tk.TOOL_REGISTRY[name]
    print(f"=== {name} ===")
    for key, val in info.items():
        print(f"  {key:12s}: {val}")
    print()

=== REVEL ===
  kind        : missense
  learning    : supervised
  signal      : random-forest ENSEMBLE of 13 scores, trained on curated pathogenic/benign labels
  circularity : HIGH (label lineage overlaps ClinVar/HGMD)
  pmid        : 27666373

=== PrimateAI ===
  kind        : missense
  learning    : semi-supervised
  signal      : deep net trained on common human/primate missense as a benign proxy
  circularity : medium
  pmid        : 30038395



## 10 · Key takeaways

> ### 🎯 Supervised meta-predictors are powerful — but you must worry about **circularity** when benchmarking against ClinVar.

1. **REVEL** is a **supervised random-forest ensemble** of 13 predictors, trained on curated
   pathogenic/benign variants. Score 0–1, higher = more pathogenic.
2. Because REVEL's training labels **share lineage with ClinVar/HGMD**, grading REVEL against
   ClinVar is **partly circular** (label leakage). It can look more accurate than it really is on
   *new* variants.
3. The **unsupervised** tools from notebook 03 (AlphaMissense, EVE, ESM1b) never saw clinical
   labels, so they give a more **independent** opinion to benchmark with.
4. REVEL is best read as a **graded** scale (Pejaver 2022: ~0.290 / 0.644 / 0.773 / 0.932 →
   increasing evidence strength), **not** a single 0.75 yes/no cut.
5. **PrimateAI** is **semi-supervised**: it learns benignity from **common human & primate**
   variants — a clever proxy that avoids clinical pathogenic labels, giving it only **medium**
   circularity.
6. Always **keep the `source` column visible** and, for real work, download the genuine
   genome-wide scores and **join on genomic coordinate** (mind CFTR's minus strand!).

➡️ **Next:** notebook **08** puts these ideas to work, benchmarking every tool against *orthogonal*
truth sets so supervised predictors aren't graded on their own homework.
